In [ ]:
!pip install -q torch-geometric
import torch
from torch_geometric.datasets import MD17
import time
import os
import shutil

print("Starting FoldPipe Data Serialization...")
start_time = time.time()
dataset = MD17(root='./data', name='aspirin')
num_graphs = len(dataset)
num_atoms = dataset[0].z.shape[0]

z_tensor = dataset[0].z 
pos_tensor = torch.zeros((num_graphs, num_atoms, 3), dtype=torch.float32)
energy_tensor = torch.zeros((num_graphs, 1), dtype=torch.float32)
force_tensor = torch.zeros((num_graphs, num_atoms, 3), dtype=torch.float32)

for i in range(num_graphs):
    data = dataset[i]
    pos_tensor[i] = data.pos
    energy_tensor[i] = data.energy
    force_tensor[i] = data.force

os.makedirs('./foldpipe_data', exist_ok=True)
torch.save({
    'z': z_tensor,
    'pos': pos_tensor,
    'energy': energy_tensor,
    'force': force_tensor
}, './foldpipe_data/aspirin_flat.pt')
print("SERIALIZATION COMPLETE")


In [ ]:
import time
import torch
import psutil
from torch.utils.data import Dataset, DataLoader
import shutil
import os

class FoldPipeDataset(Dataset):
    def __init__(self, data_path):
        print("Loading flat binary tensors...")
        data = torch.load(data_path, weights_only=True)
        self.z = data['z']
        self.pos = data['pos']
        self.energy = data['energy']
        self.force = data['force']
        self.num_samples = self.pos.shape[0]
        
    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.pos[idx], self.energy[idx], self.force[idx]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset_opt = FoldPipeDataset('./foldpipe_data/aspirin_flat.pt')

batch_size = 64
num_workers = 4

loader_opt = DataLoader(
    dataset_opt, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=num_workers,
    pin_memory=True 
)

print(f"\nStarting FoldPipe Profiling Run (Batch Size: {batch_size}, Workers: {num_workers})...")

num_batches_to_profile = 500
cpu_load_times = []
gpu_compute_times = []

if torch.cuda.is_available():
    torch.cuda.synchronize()
    
global_start = time.time()
epoch_start = time.time()

for idx, (pos, energy, force) in enumerate(loader_opt):
    if idx >= num_batches_to_profile:
        break
        
    load_end = time.time()
    cpu_load_times.append(load_end - epoch_start)
    
    pos = pos.to(device, non_blocking=True)
    energy = energy.to(device, non_blocking=True)
    force = force.to(device, non_blocking=True)
    
    if torch.cuda.is_available():
        dummy_matrix = torch.randn((batch_size, 512, 512), device=device)
        _ = torch.bmm(dummy_matrix, dummy_matrix)
        torch.cuda.synchronize()
    
    compute_end = time.time()
    gpu_compute_times.append(compute_end - load_end)
    epoch_start = time.time()

total_cpu_time = sum(cpu_load_times)
total_gpu_time = sum(gpu_compute_times)
total_runtime = total_cpu_time + total_gpu_time

print("\n" + "="*50)
print("FOLDPIPE METRICS (OPTIMIZED ARCHITECTURE)")
print("="*50)
print(f"Total Batches Processed : {num_batches_to_profile}")
print(f"Total Profiling Time    : {total_runtime:.2f} seconds")
print("-" * 50)
print(f"CPU Load Time (Idle GPU): {total_cpu_time:.2f} sec ({(total_cpu_time/total_runtime)*100:.1f}%)")
print(f"GPU Active Compute Time : {total_gpu_time:.2f} sec ({(total_gpu_time/total_runtime)*100:.1f}%)")
print("-" * 50)
print(f"System RAM Peak Used    : {psutil.virtual_memory().percent}%")
print("="*50)

# CLEANUP TO PREVENT HUGE DOWNLOADS
shutil.rmtree('./data', ignore_errors=True)
shutil.rmtree('./foldpipe_data', ignore_errors=True)
